# RideWise · Notebook 01 — Data Audit & Cleaning

**The five RideWise tables, check they join cleanly, and record the single most important fact about this dataset.**

---

### What you will learn
- How to load and profile a multi-table dataset
- How to verify referential integrity before you trust a join
- How to spot when a target variable carries no signal — and why that matters
- How to clean trip records with business-sensible filters

### How to read this notebook
Every section follows the same rhythm used throughout the project:
**the business question first**, then the data, then the method, then a
**validation check** that proves the step did what we claimed. Run the cells
top to bottom; nothing depends on hidden state.

---

## 1. The business question

Before any modelling, a data scientist has to answer a blunt question:
**can this data actually support the project?** RideWise wants to predict
churn. That is only possible if rider behaviour in the data is related to
whether riders leave. This notebook checks exactly that, and the answer
shapes every later decision.

In [1]:
# --- environment setup (run me first) ---
import sys, os
from pathlib import Path

# Make the shared pipeline importable whether you launch from notebooks/ or root
ROOT = Path.cwd()
if (ROOT / "src").exists():
    SRC = ROOT / "src"
elif (ROOT.parent / "src").exists():
    SRC = ROOT.parent / "src"
else:
    raise FileNotFoundError("Could not locate the src/ folder with ridewise_pipeline.py")
sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print("Setup OK · pipeline module at:", SRC)

Setup OK · pipeline module at: /home/claude/ridewise/src


## 2. Load the five tables

In [2]:
from ridewise_pipeline import load_raw
raw = load_raw()
for name, df in raw.items():
    print(f"{name:12s} {df.shape[0]:>7,} rows  x  {df.shape[1]:>2} cols")

riders        10,000 rows  x   8 cols
drivers        5,000 rows  x   7 cols
trips        200,000 rows  x  16 cols
sessions      50,000 rows  x   8 cols
promotions        20 rows  x  11 cols


**Worked example — peek at each table.** Always look at real rows, not just
column names. Numbers and formats tell you more than a schema does.

In [3]:
raw["riders"].head(3)

,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churn_prob,referred_by
0,R00000,2025-01-24,Bronze,34.729629,Nairobi,5.0,0.142431,R00001
1,R00001,2024-09-09,Bronze,34.571020,Nairobi,4.7,0.674161,NaN
2,R00002,2024-09-07,Bronze,47.133960,Lagos,4.2,0.510379,NaN


In [4]:
raw["trips"].head(3)

,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 23:13:48+00:14,2024-10-28 23:26:48+00:14,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 05:36:41+02:27,2025-02-17 05:52:41+02:27,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze


## 3. Referential integrity

A churn model joins trips and sessions onto riders. If trips referenced
riders that do not exist, those joins would silently drop data. **Check
before you trust.**

In [5]:
riders, trips, sessions = raw["riders"], raw["trips"], raw["sessions"]
trip_match = trips["user_id"].isin(riders["user_id"]).mean()
sess_match = sessions["rider_id"].isin(riders["user_id"]).mean()
print(f"Trips that map to a known rider:    {trip_match:.2%}")
print(f"Sessions that map to a known rider: {sess_match:.2%}")
assert trip_match == 1.0 and sess_match == 1.0, "Orphan records present!"
print("\nValidation check PASSED — every trip and session has a parent rider.")

Trips that map to a known rider:    100.00%
Sessions that map to a known rider: 100.00%

Validation check PASSED — every trip and session has a parent rider.


## 4. The crucial signal check

The `riders` table ships with a `churn_prob` column. The obvious move is to
threshold it into a yes/no label and model that. But a label is only useful
if it **relates to the features**. Let's test whether `churn_prob` depends on
anything at all.

In [6]:
prof = riders.copy()
prof["churn_flag"] = (prof["churn_prob"] > 0.5).astype(int)

print("Correlation of churn_prob with numeric attributes:")
for col in ["age", "avg_rating_given"]:
    print(f"  {col:18s}: {prof[['churn_prob', col]].corr().iloc[0,1]:+.3f}")

print("\nMean churn flag by loyalty tier:")
print(prof.groupby("loyalty_status")["churn_flag"].mean().round(3))

print("\nMean churn flag by city:")
print(prof.groupby("city")["churn_flag"].mean().round(3))

Correlation of churn_prob with numeric attributes:
  age               : -0.003
  avg_rating_given  : -0.001

Mean churn flag by loyalty tier:
loyalty_status
Bronze      0.108
Gold        0.097
Platinum    0.124
Silver      0.101
Name: churn_flag, dtype: float64

Mean churn flag by city:
city
Cairo      0.103
Lagos      0.106
Nairobi    0.110
Name: churn_flag, dtype: float64


### What this tells us

Every correlation is essentially **zero**, and churn does not vary by loyalty
tier or city. In plain terms: the supplied churn probability is **statistical
noise** — it is unrelated to who the rider is or how they behave. A model
trained to predict it would score no better than a coin flip (ROC-AUC ≈ 0.50).

> **This is the defining finding of the project.** We handle it transparently
> in Notebook 03 by constructing a *behavioural* churn label with realistic
> signal, and we document that step openly rather than pretending the raw
> data was learnable. Honesty about data quality is part of the craft.

## 5. Clean the trip records

Real trip logs contain impossible rows: zero fares, negative tips, trips that
last a fraction of a second or longer than a day. The pipeline's `clean_trips`
applies business-sensible filters and derives trip duration.

In [7]:
from ridewise_pipeline import clean_trips, clean_riders
trips_clean = clean_trips(trips)
riders_clean = clean_riders(riders)
print("Trips after cleaning:", f"{len(trips_clean):,}")
print("New columns:", [col for col in trips_clean.columns if col not in trips.columns])
trips_clean[["fare", "surge_multiplier", "tip", "duration_min"]].describe().round(2)

Trips after cleaning: 200,000
New columns: ['duration_min']


,fare,surge_multiplier,tip,duration_min
count,200000.00,200000.00,200000.00,200000.00
mean,15.40,1.14,0.47,31.96
std,6.16,0.26,1.10,15.87
min,2.97,1.00,0.00,5.00
25%,11.00,1.00,0.00,18.00
50%,14.13,1.00,0.00,32.00
75%,18.35,1.20,0.40,46.00
max,82.74,3.80,21.86,59.00


## 6. Summary

- Five tables, all joining cleanly on rider id.
- 200k trips and 50k sessions give plenty of behavioural depth.
- **The raw churn target carries no signal** — the key constraint we design around.
- Trips are cleaned with sanity filters and a derived `duration_min`.

**Next:** Notebook 02 explores the cleaned data visually before we engineer features.